# Case study: Amazon reviews
**Web and Text Analytics 2026-27 · Week 5 · From tokens to vectors**

Sentiment analysis on the Kaggle dataset *Amazon Reviews: Unlocked Mobile Phones*: 413,840 reviews of 4,410 phones from 384 brands.
From the text of a review we predict whether its author was positive or negative. The label comes for free: the star rating.

Each section below is one **Notebook · Step** of the slides, in the same order and with the same code.

Data: https://www.kaggle.com/datasets/PromptCloudHQ/amazon-reviews-unlocked-mobile-phones · the next cell downloads it directly from Kaggle, so there is nothing to download by hand. If you already have `Amazon_Unlocked_Mobile.csv`, put it in the same folder as this notebook (or upload it to Colab) and it is used instead.
Checked with scikit-learn 1.9.1, pandas 3.0 and numpy 2.5.

## Step 1 · Review data
Import the data. Sample the data to speed up computation: we keep 10% of the reviews.
All numbers on the slides come from this 10% sample. Comment out the `sample` line to use all the reviews: the numbers will change.

In [ ]:
#@title ▶ Load the reviews (about 400,000 rows, 1–2 minutes)
# Option A: download from Kaggle. Option B: upload Amazon_Unlocked_Mobile.csv to Colab (Files panel) and it is used instead.
import os, glob
import pandas as pd
import numpy as np

if os.path.exists("Amazon_Unlocked_Mobile.csv"):
    csv_path = "Amazon_Unlocked_Mobile.csv"
else:
    import kagglehub
    folder = kagglehub.dataset_download("PromptCloudHQ/amazon-reviews-unlocked-mobile-phones")
    csv_path = glob.glob(os.path.join(folder, "*.csv"))[0]

df = pd.read_csv(csv_path)                  # 413,840 reviews
df.index.name = "review_id"                 # the original row number identifies a review in every later week
df = df.sample(frac=0.1, random_state=10)   # 10%: 41,384 reviews
df.head()

### 5, 1 and 3 star reviews


In [ ]:
df[(df["Rating"] == 5)].head()   # praise: great, best, love, excellent, no problems

In [ ]:
df[(df["Rating"] == 1)].head()   # complaints, often with a negation

In [ ]:
df[(df["Rating"] == 3)].head()   # mixed and ambiguous: we will drop them

## Step 2 · Data preprocessing
Drop missing values. Remove any 'neutral' ratings (3 stars). Create a new variable `Positively Rated`: 1 for 4 and 5 stars, 0 for 1 and 2 stars.

In [ ]:
df.dropna(inplace=True)   # drop missing values
df = df[df["Rating"] != 3]   # remove neutral ratings
df["Positively Rated"] = np.where(df["Rating"] > 3, 1, 0)
print(len(df), "reviews left")   # 30,736 reviews left
df.head()

### Most ratings are positive
Imbalanced classes: about three positive reviews for every negative one. A model that always answers "positive" is right 75% of the time
without reading a word, so we report **AUC**: how well the model ranks positive reviews above negative ones, whatever the balance.

In [ ]:
df["Positively Rated"].mean()

## Step 3 · Train – test
Predictor X: the text of the review (`Reviews`). Response Y: its sentiment (`Positively Rated`).
By default `train_test_split` keeps 25% of the reviews for the test set and 75% for the training set.

**Fit everything on `X_train` only**: the vocabulary, the weights. `X_test` is used once, to score the model.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df["Reviews"],
                                                    df["Positively Rated"],
                                                    random_state=0)
print("X_train first entry:", X_train.iloc[0])
print("X_train shape:", X_train.shape)

## Step 4 · CountVectorizer
`CountVectorizer` is scikit-learn's bag of words: it converts a collection of text documents into a matrix of token counts.
Fitting it tokenizes the training reviews (runs of at least two letters or digits, lowercased) and builds the vocabulary.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vect = CountVectorizer().fit(X_train)   # fit on the training reviews only
vect

### Vocabulary
`get_feature_names_out()` returns the vocabulary (it was `get_feature_names()` in older versions of scikit-learn).
Every 2000th word gives a small sense of it: words with numbers, misspellings, other languages.

In [ ]:
vect.get_feature_names_out()[::2000]

In [ ]:
len(vect.get_feature_names_out())

## Step 5 · transform()
`transform` turns the reviews of `X_train` into a document-term matrix: one row per review, one column per word of the vocabulary,
each cell the number of times the word appears in the review. It is a SciPy sparse matrix: only the non-zero cells are stored.

In [ ]:
# transform the documents in the training data to a document-term matrix
X_train_vectorized = vect.transform(X_train)

X_train_vectorized

In [ ]:
# share of the cells that are zero
1 - X_train_vectorized.nnz / (X_train_vectorized.shape[0] * X_train_vectorized.shape[1])

## Step 6 · Machine learning
`LogisticRegression` works well for high-dimensional sparse data. `max_iter=2000` gives the solver enough iterations to converge.

AUC needs **scores, not labels**: `predict_proba` gives the probability of "positive" for every review; `predict()` would give only 0/1,
and `roc_auc_score` on 0/1 labels returns balanced accuracy, not AUC.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

model = LogisticRegression(max_iter=2000)
model.fit(X_train_vectorized, y_train)   # numeric data, not text

scores = model.predict_proba(vect.transform(X_test))[:, 1]   # probability of "positive"
print("AUC:", roc_auc_score(y_test, scores))

### Coefficients of the model
The words the model has connected to negative (smallest coefficients) and positive (largest coefficients) reviews.

In [ ]:
feature_names = np.array(vect.get_feature_names_out())   # get the feature names as a numpy array
sorted_coef_index = model.coef_[0].argsort()   # sort the coefficients
print("Smallest Coefs:", feature_names[sorted_coef_index[:10]])
print("Largest Coefs:", feature_names[sorted_coef_index[:-11:-1]])

## Step 7 · TF-IDF
Features with high TF-IDF are used frequently within specific documents, but rarely across all documents.
Features with low TF-IDF are either used commonly across all documents, or used rarely and only in long documents.

`min_df` (in `CountVectorizer` and `TfidfVectorizer`): the minimum number of documents a token must appear in to become part of the vocabulary.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer(min_df=5).fit(X_train)   # fit the TfidfVectorizer to the training data
len(vect.get_feature_names_out())

### Machine learning with TF-IDF
Two things change at once here, TF-IDF and `min_df=5`, so the difference in AUC cannot be credited to TF-IDF alone.

In [ ]:
X_train_vectorized = vect.transform(X_train)

model = LogisticRegression(max_iter=2000)
model.fit(X_train_vectorized, y_train)

scores = model.predict_proba(vect.transform(X_test))[:, 1]
print("AUC:", roc_auc_score(y_test, scores))

### Smallest and largest TF-IDF
For every word, its largest TF-IDF weight in any training review.

In [ ]:
feature_names = np.array(vect.get_feature_names_out())
sorted_tfidf_index = X_train_vectorized.max(0).toarray()[0].argsort()   # largest weight of each word in any review
print("Smallest tfidf:", feature_names[sorted_tfidf_index[:10]])
print("Largest tfidf:", feature_names[sorted_tfidf_index[:-11:-1]])

### Coefficients

In [ ]:
sorted_coef_index = model.coef_[0].argsort()
print("Smallest Coefs:", feature_names[sorted_coef_index[:10]])
print("Largest Coefs:", feature_names[sorted_coef_index[:-11:-1]])

## Step 8 · N-grams
Bag of words loses the order of the words: 'not an issue, phone is working' is seen the same as 'an issue, phone is not working'.
We can add sequences of words, known as n-grams. Bigrams count pairs of adjacent words and give features such as 'is working' versus 'not working'.

In [ ]:
# these reviews are treated the same by our current model
print(model.predict(vect.transform(['not an issue, phone is working',
                                    'an issue, phone is not working'])))

In [ ]:
# document frequency of 5 and extracting 1-grams and 2-grams
vect = CountVectorizer(min_df=5, ngram_range=(1, 2)).fit(X_train)
X_train_vectorized = vect.transform(X_train)
len(vect.get_feature_names_out())

In [ ]:
model = LogisticRegression(max_iter=2000)
model.fit(X_train_vectorized, y_train)
scores = model.predict_proba(vect.transform(X_test))[:, 1]
print("AUC:", roc_auc_score(y_test, scores))

In [ ]:
feature_names = np.array(vect.get_feature_names_out())
sorted_coef_index = model.coef_[0].argsort()
print("Smallest Coefs:", feature_names[sorted_coef_index[:10]])
print("Largest Coefs:", feature_names[sorted_coef_index[:-11:-1]])

Bigrams bring back a little of the word order that the bag of words throws away.

In [ ]:
# these reviews are now correctly identified
print(model.predict(vect.transform(['not an issue, phone is working',
                                    'an issue, phone is not working'])))